# Day 5 — **Personal AI Knowledge Worker (RAG) — Advanced Connectors** — Colab‑Ready (T4 GPU)

This notebook implements the **advanced ideas** from Lesson 127:

- **Google Workspace**: read **Docs / Sheets / Slides** via Google Drive + Docs/Sheets/Slides APIs (read‑only)
- **Gmail**: pull emails (read‑only) with a search query (e.g., `from:alice project X`)
- **Slack**: ingest channel history (read‑only) using a **Bot token** (no write scope)
- **MS Office**: parse `.docx`, `.pptx`, `.xlsx` (plus `.pdf`, `.txt`, `.md`, `.csv`) locally
- Unified **RAG** pipeline: Chroma/FAISS vector store, local/OpenAI embeddings, local/frontier LLMs, Gradio UI

> ⚠️ **Privacy**: Use **local embeddings + local LLM** to keep text local. If you choose OpenAI
embeddings/LLM, snippets & prompts go to their API. All cloud connectors here are **read‑only** and require your explicit authorization.

## 0) Runtime check

In [1]:
import os, sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())
try:
    import torch
    print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch not installed yet:", e)

Python: 3.11.13 | packaged by conda-forge | (main, Jun  4 2025, 14:48:23) [GCC 13.3.0]
Platform: Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.39
Torch: 2.7.1 | CUDA available: True
GPU: NVIDIA GeForce RTX 3060


## 1) Installs (Colab‑friendly)

In [2]:
# # Core RAG & vector stores
# !pip install -q -U langchain langchain-community langchain-openai langchain-text-splitters
# !pip install -q -U chromadb faiss-cpu

# # Embeddings & LLMs
# !pip install -q -U sentence-transformers transformers accelerate bitsandbytes tiktoken

# File loaders
!pip install -q -U pypdf docx2txt python-pptx openpyxl

# # UI & utils
# !pip install -q -U gradio pandas beautifulsoup4 python-dateutil

# # Frontier clients (optional)
# !pip install -q -U openai==1.*

# Google APIs (OAuth client + services)
!pip install -q -U google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client

# Slack SDK (read-only via Bot token)
!pip install -q -U slack_sdk

## 2) Imports & device

In [3]:
import os, re, io, json, glob, shutil, textwrap, tempfile, warnings, base64
import datetime as dt
from dataclasses import dataclass
from typing import List, Dict, Any, Optional, Tuple

import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from dateutil import parser as dateparser

# LangChain pieces
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma, FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings

# LLM (local) – transformers for full control
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Frontier client (optional)
try:
    from openai import OpenAI
except Exception:
    OpenAI = None

# File parsing helpers
from pypdf import PdfReader
import docx2txt
from pptx import Presentation
import openpyxl

# Google APIs
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# Slack
from slack_sdk import WebClient
from slack_sdk.errors import SlackApiError

import gradio as gr

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 3) (Optional) Mount Google Drive

In [4]:
# try:
#     from google.colab import drive  # type: ignore
#     drive.mount('/content/drive')
#     print("Drive mounted at /content/drive")
# except Exception:
#     print("Not running in Colab or Drive not available.")

## 4) Configuration & paths

In [5]:
# Chroma persistence
DEFAULT_PERSIST_DIR = "personal-knowledge-base-advanced" # "/content/chroma_kb_advanced"
os.makedirs(DEFAULT_PERSIST_DIR, exist_ok=True)

# Frontier API (optional)
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", None)
openai_client = None
if OPENAI_API_KEY and OpenAI is not None:
    try:
        openai_client = OpenAI(api_key=OPENAI_API_KEY)
        print("✅ OpenAI client ready")
    except Exception as e:
        print("⚠️ OpenAI client init failed:", e)

# Open‑source LLM default
DEFAULT_LOCAL_LLM = "microsoft/Phi-3-mini-4k-instruct"  # ~3.8B; fits T4 in 4‑bit

# Where to store OAuth tokens & client secrets in Colab
GOOGLE_CLIENT_SECRETS = "/content/google_client_secrets.json"  # upload your OAuth client JSON here
TOKEN_DRIVE = "/content/token_drive.json"
TOKEN_GMAIL = "/content/token_gmail.json"

# Slack
SLACK_BOT_TOKEN = os.environ.get("SLACK_BOT_TOKEN", "")  # or paste in UI

## 5) Local file loaders (MS Office + common text formats)
Supported: **.txt, .md, .pdf, .docx, .pptx, .xlsx, .csv**

In [6]:
def load_text_file(path: str) -> str:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def load_pdf(path: str) -> str:
    try:
        reader = PdfReader(path)
        texts = [page.extract_text() or "" for page in reader.pages]
        return "\n".join(texts)
    except Exception:
        return ""

def load_docx(path: str) -> str:
    try:
        return docx2txt.process(path) or ""
    except Exception:
        return ""

def load_pptx(path: str) -> str:
    try:
        prs = Presentation(path)
        texts = []
        for slide in prs.slides:
            for shape in slide.shapes:
                if hasattr(shape, "text") and shape.text:
                    texts.append(shape.text)
        return "\n".join(texts)
    except Exception:
        return ""

def load_xlsx(path: str, max_cells: int = 20000) -> str:
    try:
        wb = openpyxl.load_workbook(path, data_only=True, read_only=True)
        parts = []
        cells_read = 0
        for ws in wb.worksheets:
            parts.append(f"## Sheet: {ws.title}")
            for row in ws.iter_rows(values_only=True):
                if row is None:
                    continue
                line = ",".join("" if v is None else str(v) for v in row)
                parts.append(line)
                cells_read += len(row)
                if cells_read >= max_cells:
                    parts.append("... (truncated)")
                    break
            if cells_read >= max_cells:
                break
        return "\n".join(parts)
    except Exception:
        return ""

def load_csv(path: str, max_chars: int = 150_000) -> str:
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()[:max_chars]
    except Exception:
        return ""

SUPPORTED_SUFFIXES = {".txt", ".md", ".pdf", ".docx", ".pptx", ".xlsx", ".csv"}

def read_any(path: str) -> str:
    ext = os.path.splitext(path)[1].lower()
    if ext in {".txt", ".md"}:
        return load_text_file(path)
    if ext == ".pdf":
        return load_pdf(path)
    if ext == ".docx":
        return load_docx(path)
    if ext == ".pptx":
        return load_pptx(path)
    if ext == ".xlsx":
        return load_xlsx(path)
    if ext == ".csv":
        return load_csv(path)
    return ""

## 6) Chunking

In [7]:
def chunk_documents(docs: List[Dict[str, Any]], chunk_size: int = 800, chunk_overlap: int = 200):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    out_docs = []
    for d in docs:
        for chunk in splitter.split_text(d["text"]):
            out_docs.append({
                "page_content": chunk,
                "metadata": d.get("metadata", {}).copy()
            })
    return out_docs

## 7) Embeddings: OpenAI or Local (Sentence‑Transformers)

In [8]:
@dataclass
class EmbedderCfg:
    kind: str  # "openai" | "local-minilm" | "local-bge"
    model: str

def create_embedder(cfg: EmbedderCfg):
    if cfg.kind == "openai":
        if not OPENAI_API_KEY or OpenAIEmbeddings is None:
            raise RuntimeError("OpenAI API key not set or OpenAIEmbeddings missing.")
        return OpenAIEmbeddings(model=cfg.model)
    elif cfg.kind == "local-minilm":
        return HuggingFaceEmbeddings(model_name=cfg.model, encode_kwargs={"normalize_embeddings": True})
    elif cfg.kind == "local-bge":
        return HuggingFaceEmbeddings(model_name=cfg.model, encode_kwargs={"normalize_embeddings": True})
    else:
        raise ValueError("Unknown embedder kind")

## 8) Vector stores: **Chroma** (persistent) or **FAISS** (in‑memory, savable)

In [9]:
@dataclass
class VSHandle:
    kind: str  # "chroma" | "faiss"
    handle: Any

def build_vector_store(docs, embedder, kind: str = "chroma", persist_dir: str = DEFAULT_PERSIST_DIR, collection_name: str = "personal_kb_adv"):
    if kind == "chroma":
        vs = Chroma.from_documents(
            documents=[type("Doc", (), d) for d in docs],
            embedding=embedder,
            persist_directory=persist_dir,
            collection_name=collection_name
        )
        return VSHandle(kind="chroma", handle=vs)
    elif kind == "faiss":
        vs = FAISS.from_documents(
            documents=[type("Doc", (), d) for d in docs],
            embedding=embedder
        )
        return VSHandle(kind="faiss", handle=vs)
    else:
        raise ValueError("Unknown vector store kind")

def save_faiss(vs_handle: VSHandle, path: str):
    assert vs_handle.kind == "faiss"
    vs_handle.handle.save_local(path)

def load_faiss(path: str, embedder) -> VSHandle:
    vs = FAISS.load_local(path, embeddings=embedder, allow_dangerous_deserialization=True)
    return VSHandle(kind="faiss", handle=vs)

## 9) Open‑source LLM loader (4‑bit on T4)

In [10]:
_tok = None
_mdl = None

def load_local_llm(model_id: str = DEFAULT_LOCAL_LLM):
    global _tok, _mdl
    if _mdl is not None and getattr(_mdl, "name_or_path", None) == model_id:
        return _tok, _mdl
    print(f"Loading local LLM: {model_id}")
    kwargs = {}
    if DEVICE == "cuda":
        try:
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16,
            )
            kwargs["device_map"] = "auto"
            kwargs["torch_dtype"] = torch.bfloat16
        except Exception as e:
            print("4-bit load failed; falling back:", e)
            kwargs["device_map"] = "auto"
    _tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    if _tok.pad_token is None:
        _tok.pad_token = _tok.eos_token
    _mdl = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    return _tok, _mdl

## 10) RAG core: retrieval → prompt assembly → generation

In [11]:
SYSTEM_PROMPT = (
    "You are a helpful, concise assistant. Answer using only the provided context. "
    "If the answer is not in the context, say you don't know. "
    "Cite sources with (Source: filename or path)."
)

def build_context_str(docs: List[Dict[str, Any]], max_chars: int = 8_000) -> Tuple[str, List[Dict[str, Any]]]:
    pieces = []
    used = []
    total = 0
    for d in docs:
        meta = d.get("metadata", {})
        fname = meta.get("source", meta.get("path", "unknown"))
        snippet = d.get("page_content", "")
        snippet = snippet[:1500]
        piece = f"[Source: {fname}]\n{snippet}"
        if total + len(piece) > max_chars:
            break
        pieces.append(piece)
        used.append({"source": fname, "snippet": snippet})
        total += len(piece)
    return "\n\n".join(pieces), used

def retrieve_docs(vs_handle: VSHandle, query: str, k: int = 12):
    retriever = vs_handle.handle.as_retriever(search_kwargs={"k": int(k)})
    docs = retriever.get_relevant_documents(query)
    out = []
    for d in docs:
        out.append({"page_content": d.page_content, "metadata": dict(d.metadata or {})})
    return out

def generate_frontier_answer(question: str, context: str, model: str = "gpt-4o-mini", temperature: float = 0.2, max_tokens: int = 512) -> str:
    if openai_client is None:
        raise RuntimeError("OpenAI client not configured (set OPENAI_API_KEY).")
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    try:
        resp = openai_client.responses.create(
            model=model,
            input=messages,
            temperature=temperature,
            max_output_tokens=max_tokens,
        )
        return resp.output_text.strip()
    except Exception:
        chat = openai_client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
        )
        return chat.choices[0].message.content.strip()

def generate_local_answer(question: str, context: str, model_id: str = DEFAULT_LOCAL_LLM, temperature: float = 0.2, max_new_tokens: int = 512) -> str:
    tok, mdl = load_local_llm(model_id=model_id)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    try:
        input_ids = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
    except Exception:
        prompt = f"{SYSTEM_PROMPT}\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:"
        input_ids = tok.encode(prompt, return_tensors="pt")
    if DEVICE == "cuda":
        input_ids = input_ids.to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(
            input_ids=input_ids,
            max_new_tokens=int(max_new_tokens),
            temperature=float(temperature),
            do_sample=True,
            top_p=0.95,
            pad_token_id=tok.eos_token_id,
        )
    gen = out[0, input_ids.shape[1]:]
    text = tok.decode(gen, skip_special_tokens=True).strip()
    return text

## 11) **Google OAuth** helpers (Drive/Docs/Sheets/Slides & Gmail)

In [12]:
# Scopes: least-privilege read-only
SCOPES_DRIVE = [
    "https://www.googleapis.com/auth/drive.readonly",
    "https://www.googleapis.com/auth/documents.readonly",
    "https://www.googleapis.com/auth/spreadsheets.readonly",
    "https://www.googleapis.com/auth/presentations.readonly",
]
SCOPES_GMAIL = ["https://www.googleapis.com/auth/gmail.readonly"]

def ensure_google_auth(token_path: str, scopes: List[str]) -> Credentials:
    creds = None
    if os.path.exists(token_path):
        creds = Credentials.from_authorized_user_file(token_path, scopes)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            try:
                creds.refresh(Request())  # type: ignore
            except Exception:
                creds = None
        if not creds:
            if not os.path.exists(GOOGLE_CLIENT_SECRETS):
                raise RuntimeError(
                    f"Missing client secret JSON at {GOOGLE_CLIENT_SECRETS}. "
                    "Download from Google Cloud Console (OAuth 2.0 Client IDs, Desktop app) and upload it."
                )
            flow = InstalledAppFlow.from_client_secrets_file(GOOGLE_CLIENT_SECRETS, scopes=scopes)
            # In Colab, run_console often works better than run_local_server
            creds = flow.run_console()
        with open(token_path, "w") as token:
            token.write(creds.to_json())
    return creds

## 12) Google Workspace connectors (Drive + Docs/Sheets/Slides)

In [13]:
MIME_GOOGLE_DOC = "application/vnd.google-apps.document"
MIME_GOOGLE_SHEET = "application/vnd.google-apps.spreadsheet"
MIME_GOOGLE_SLIDE = "application/vnd.google-apps.presentation"

def gdrive_list_files(query: str, max_files: int = 50) -> List[Dict[str, Any]]:
    creds = ensure_google_auth(TOKEN_DRIVE, SCOPES_DRIVE)
    service = build("drive", "v3", credentials=creds, cache_discovery=False)
    items = []
    page_token = None
    while True:
        resp = service.files().list(
            q=query,
            pageSize=min(max_files, 100),
            fields="nextPageToken, files(id, name, mimeType, modifiedTime)",
            pageToken=page_token
        ).execute()
        items.extend(resp.get("files", []))
        page_token = resp.get("nextPageToken")
        if not page_token or len(items) >= max_files:
            break
    return items[:max_files]

def gdoc_export_txt(file_id: str) -> str:
    creds = ensure_google_auth(TOKEN_DRIVE, SCOPES_DRIVE)
    service = build("drive", "v3", credentials=creds, cache_discovery=False)
    try:
        data = service.files().export(fileId=file_id, mimeType="text/plain").execute()
        return data.decode("utf-8", errors="ignore") if isinstance(data, (bytes, bytearray)) else str(data)
    except HttpError:
        return ""

def gsheet_export_text(file_id: str, max_cells: int = 20000) -> str:
    creds = ensure_google_auth(TOKEN_DRIVE, SCOPES_DRIVE)
    sh = build("sheets", "v4", credentials=creds, cache_discovery=False).spreadsheets()
    meta = sh.get(spreadsheetId=file_id).execute()
    parts, cells = [], 0
    for sheet in meta.get("sheets", []):
        title = sheet["properties"]["title"]
        parts.append(f"## Sheet: {title}")
        rng = f"'{title}'!A1:Z9999"
        values = sh.values().get(spreadsheetId=file_id, range=rng).execute().get("values", [])
        for row in values:
            parts.append(",".join(row))
            cells += len(row)
            if cells >= max_cells:
                parts.append("... (truncated)")
                return "\n".join(parts)
    return "\n".join(parts)

def gslides_export_text(file_id: str) -> str:
    # Export to PPTX then parse text locally
    creds = ensure_google_auth(TOKEN_DRIVE, SCOPES_DRIVE)
    drive = build("drive", "v3", credentials=creds, cache_discovery=False)
    try:
        data = drive.files().export(fileId=file_id, mimeType="application/vnd.openxmlformats-officedocument.presentationml.presentation").execute()
        tmp = tempfile.mkdtemp()
        path = os.path.join(tmp, "slide.pptx")
        with open(path, "wb") as f:
            f.write(data)
        return load_pptx(path)
    except HttpError:
        return ""

def collect_from_gworkspace(query: str, include_docs=True, include_sheets=True, include_slides=True, max_files: int = 50):
    q_parts = []
    mt_filters = []
    if include_docs:
        mt_filters.append(f"mimeType='{MIME_GOOGLE_DOC}'")
    if include_sheets:
        mt_filters.append(f"mimeType='{MIME_GOOGLE_SHEET}'")
    if include_slides:
        mt_filters.append(f"mimeType='{MIME_GOOGLE_SLIDE}'")
    if mt_filters:
        q_parts.append("(" + " or ".join(mt_filters) + ")")
    if query:
        q_parts.append(f"name contains '{query.replace("'","\'")}'")
    q = " and ".join(q_parts) if q_parts else ""
    files = gdrive_list_files(q, max_files=max_files)
    docs = []
    for f in files:
        fid, name, mt = f["id"], f["name"], f["mimeType"]
        text = ""
        if mt == MIME_GOOGLE_DOC:
            text = gdoc_export_txt(fid)
        elif mt == MIME_GOOGLE_SHEET:
            text = gsheet_export_text(fid)
        elif mt == MIME_GOOGLE_SLIDE:
            text = gslides_export_text(fid)
        if text:
            docs.append({"text": text, "metadata": {"source": f"gdrive://{name}", "mimeType": mt}})
    return docs, len(files)

SyntaxError: unterminated string literal (detected at line 76) (2114049048.py, line 76)

## 13) Gmail connector (read‑only)

In [ ]:
def gmail_list_messages(query: str, max_results: int = 50) -> List[str]:
    creds = ensure_google_auth(TOKEN_GMAIL, SCOPES_GMAIL)
    svc = build("gmail", "v1", credentials=creds, cache_discovery=False)
    msgs = []
    page_token = None
    while True:
        resp = svc.users().messages().list(userId="me", q=query, maxResults=min(max_results, 100), pageToken=page_token).execute()
        for m in resp.get("messages", []):
            msgs.append(m["id"])
        page_token = resp.get("nextPageToken")
        if not page_token or len(msgs) >= max_results:
            break
    return msgs[:max_results]

def _decode_gmail_part(part):
    data = part.get("body", {}).get("data")
    if not data: 
        return ""
    decoded = base64.urlsafe_b64decode(data.encode("utf-8")).decode("utf-8", errors="ignore")
    if part.get("mimeType") == "text/html":
        return BeautifulSoup(decoded, "html.parser").get_text(separator="\n")
    return decoded

def gmail_fetch_message_text(msg_id: str) -> Tuple[str, Dict[str,str]]:
    creds = ensure_google_auth(TOKEN_GMAIL, SCOPES_GMAIL)
    svc = build("gmail", "v1", credentials=creds, cache_discovery=False)
    msg = svc.users().messages().get(userId="me", id=msg_id, format="full").execute()
    headers = {h["name"].lower(): h["value"] for h in msg.get("payload", {}).get("headers", [])}
    frm = headers.get("from", "")
    subj = headers.get("subject", "")
    date = headers.get("date", "")
    payload = msg.get("payload", {})
    parts = []
    def walk(p):
        if "parts" in p:
            for sp in p["parts"]:
                walk(sp)
        else:
            parts.append(_decode_gmail_part(p))
    walk(payload)
    text = "\n".join([p for p in parts if p]).strip()
    meta = {"from": frm, "subject": subj, "date": date}
    return text, meta

def collect_from_gmail(query: str, max_messages: int = 50):
    ids = gmail_list_messages(query, max_results=max_messages)
    docs = []
    for mid in ids:
        txt, meta = gmail_fetch_message_text(mid)
        if txt:
            src = f"gmail://{meta.get('from','unknown')} — {meta.get('subject','(no subject)')}"
            docs.append({"text": txt, "metadata": {"source": src, **meta}})
    return docs, len(ids)

## 14) Slack connector (read‑only, Bot token)

In [ ]:
def slack_get_channel_id(client: WebClient, name_or_id: str) -> Optional[str]:
    if name_or_id.startswith("C") or name_or_id.startswith("G"):
        return name_or_id
    try:
        res = client.conversations_list(limit=1000)
        chans = res.get("channels", [])
        for ch in chans:
            if ch.get("name") == name_or_id.lstrip("#"):
                return ch.get("id")
    except SlackApiError as e:
        print("Slack API error:", e.response["error"])
    return None

def slack_fetch_history(token: str, channel: str, oldest: Optional[str] = None, latest: Optional[str] = None, max_messages: int = 1000):
    client = WebClient(token=token)
    chan_id = slack_get_channel_id(client, channel)
    if not chan_id:
        raise RuntimeError(f"Channel not found: {channel}")
    oldest_ts = str(dateparser.parse(oldest).timestamp()) if oldest else None
    latest_ts = str(dateparser.parse(latest).timestamp()) if latest else None
    msgs = []
    cursor = None
    fetched = 0
    while True:
        try:
            res = client.conversations_history(channel=chan_id, limit=min(200, max_messages - fetched), oldest=oldest_ts, latest=latest_ts, cursor=cursor)
        except SlackApiError as e:
            raise RuntimeError(f"Slack error: {e.response['error']}")
        for m in res.get("messages", []):
            txt = m.get("text", "")
            if txt:
                ts = float(m.get("ts", "0"))
                dt_str = dt.datetime.utcfromtimestamp(ts).isoformat() + "Z"
                msgs.append(f"[{dt_str}] {txt}")
                fetched += 1
                if fetched >= max_messages:
                    break
        cursor = res.get("response_metadata", {}).get("next_cursor")
        if not cursor or fetched >= max_messages:
            break
    text = "\n".join(msgs)
    return [{"text": text, "metadata": {"source": f"slack://{channel}", "count": fetched}}], fetched

## 15) Indexing & Chat helpers (merge all sources)

In [ ]:
# Session state
_vs: Optional[VSHandle] = None
_embedder_cfg: Optional[EmbedderCfg] = None
_embedder = None
_persist_dir = DEFAULT_PERSIST_DIR
_collection_name = "personal_kb_adv"
_store_kind = "chroma"

def scan_folder(base_dir: str) -> List[str]:
    paths = []
    for ext in SUPPORTED_SUFFIXES:
        paths.extend(glob.glob(os.path.join(base_dir, f"**/*{ext}"), recursive=True))
    return sorted(list(set(paths)))

def ingest_local(base_dir: str, uploaded_files, chunk_size: int, chunk_overlap: int, store_kind: str, persist_dir: str, collection_name: str, embedder_kind: str, embedder_model: str):
    global _vs, _embedder_cfg, _embedder, _persist_dir, _collection_name, _store_kind
    _persist_dir = persist_dir or DEFAULT_PERSIST_DIR
    _collection_name = collection_name or "personal_kb_adv"
    _store_kind = store_kind

    # choose embedder
    kind_map = {
        "OpenAI (text-embedding-3-small)": ("openai", "text-embedding-3-small"),
        "Local (all-MiniLM-L6-v2)": ("local-minilm", "sentence-transformers/all-MiniLM-L6-v2"),
        "Local (bge-small-en-v1.5)": ("local-bge", "BAAI/bge-small-en-v1.5"),
    }
    if embedder_kind in kind_map:
        e_kind, e_model = kind_map[embedder_kind]
    else:
        e_kind, e_model = embedder_kind, embedder_model
    _embedder_cfg = EmbedderCfg(kind=e_kind, model=e_model)
    _embedder = create_embedder(_embedder_cfg)

    files = []
    if base_dir and os.path.isdir(base_dir):
        files.extend(scan_folder(base_dir))
    up_dir = tempfile.mkdtemp(prefix="uploads_")
    if uploaded_files:
        for uf in uploaded_files:
            src = uf.name if hasattr(uf, "name") else str(uf)
            if os.path.isfile(src):
                dst = os.path.join(up_dir, os.path.basename(src))
                shutil.copyfile(src, dst)
                files.append(dst)
    files = [p for p in files if os.path.splitext(p)[1].lower() in SUPPORTED_SUFFIXES]
    if not files:
        return "⚠️ No supported files found.", None

    docs = []
    for p in sorted(set(files)):
        txt = read_any(p)
        if not txt:
            continue
        docs.append({"text": txt, "metadata": {"source": p}})
    if not docs:
        return "⚠️ No text could be extracted.", None

    chunks = chunk_documents(docs, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    _vs = build_vector_store(chunks, _embedder, kind=store_kind, persist_dir=_persist_dir, collection_name=_collection_name)
    if store_kind == "chroma":
        _vs.handle.persist()
    return f"✅ Indexed {len(files)} files into {store_kind.upper()} with {len(chunks)} chunks.", len(chunks)

def ingest_gworkspace(query: str, include_docs: bool, include_sheets: bool, include_slides: bool, max_files: int, chunk_size: int, chunk_overlap: int):
    if _embedder is None:
        return "⚠️ Choose embeddings & build any index first (local tab) to initialize the embedder.", None
    docs, count = collect_from_gworkspace(query, include_docs, include_sheets, include_slides, max_files=max_files)
    if not docs:
        return "⚠️ No Google Workspace docs found or exported.", None
    chunks = chunk_documents(docs, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    if _vs is None:
        # create a fresh store with current embedder
        vs = build_vector_store(chunks, _embedder, kind=_store_kind, persist_dir=_persist_dir, collection_name=_collection_name)
        globals()["_vs"] = vs
    else:
        _vs.handle.add_documents([type("Doc", (), d) for d in chunks])
    if _store_kind == "chroma":
        _vs.handle.persist()
    return f"✅ Added {len(chunks)} chunks from {count} Google files.", len(chunks)

def ingest_gmail(query: str, max_messages: int, chunk_size: int, chunk_overlap: int):
    if _embedder is None:
        return "⚠️ Choose embeddings & build any index first (local tab) to initialize the embedder.", None
    docs, count = collect_from_gmail(query, max_messages=max_messages)
    if not docs:
        return "⚠️ No Gmail messages found.", None
    chunks = chunk_documents(docs, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    if _vs is None:
        vs = build_vector_store(chunks, _embedder, kind=_store_kind, persist_dir=_persist_dir, collection_name=_collection_name)
        globals()["_vs"] = vs
    else:
        _vs.handle.add_documents([type("Doc", (), d) for d in chunks])
    if _store_kind == "chroma":
        _vs.handle.persist()
    return f"✅ Added {len(chunks)} chunks from {count} Gmail messages.", len(chunks)

def ingest_slack(token: str, channel: str, oldest: str, latest: str, max_messages: int, chunk_size: int, chunk_overlap: int):
    if _embedder is None:
        return "⚠️ Choose embeddings & build any index first (local tab) to initialize the embedder.", None
    if not token:
        return "⚠️ Provide a Slack Bot token.", None
    try:
        docs, count = slack_fetch_history(token, channel, oldest=oldest, latest=latest, max_messages=max_messages)
    except Exception as e:
        return f"⚠️ Slack error: {e}", None
    if not docs:
        return "⚠️ No Slack messages found.", None
    chunks = chunk_documents(docs, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    if _vs is None:
        vs = build_vector_store(chunks, _embedder, kind=_store_kind, persist_dir=_persist_dir, collection_name=_collection_name)
        globals()["_vs"] = vs
    else:
        _vs.handle.add_documents([type("Doc", (), d) for d in chunks])
    if _store_kind == "chroma":
        _vs.handle.persist()
    return f"✅ Added {len(chunks)} chunks from Slack channel '{channel}'.", len(chunks)

def clear_index(store_kind: str, persist_dir: str):
    global _vs
    if store_kind == "chroma":
        try:
            if os.path.isdir(persist_dir):
                shutil.rmtree(persist_dir)
            os.makedirs(persist_dir, exist_ok=True)
        except Exception as e:
            return f"⚠️ Failed to clear Chroma dir: {e}"
    _vs = None
    return "✅ Cleared index."

def ask(question: str, top_k: int, model_path: str, use_frontier: bool, frontier_model: str, temperature: float, max_tokens: int, show_prompt: bool):
    if not question or not question.strip():
        return None, "⚠️ Enter a question.", None, None
    if _vs is None:
        return None, "⚠️ Index is empty. Ingest sources first.", None, None
    docs = retrieve_docs(_vs, question, k=int(top_k))
    ctx, used = build_context_str(docs)
    prompt_preview = f"{SYSTEM_PROMPT}\n\nContext (top {top_k}):\n{ctx}\n\nQuestion: {question}"
    try:
        if use_frontier:
            ans = generate_frontier_answer(question, ctx, model=frontier_model, temperature=float(temperature), max_tokens=int(max_tokens))
        else:
            ans = generate_local_answer(question, ctx, model_id=model_path, temperature=float(temperature), max_new_tokens=int(max_tokens))
    except Exception as e:
        return None, f"⚠️ Generation error: {e}", prompt_preview if show_prompt else None, used
    return ans, None, (prompt_preview if show_prompt else None), used

## 16) Gradio App (multi‑source ingestion + chat)

In [ ]:
def format_sources(used):
    if not used:
        return ""
    lines = []
    for u in used:
        src = u.get("source", "unknown")
        snippet = (u.get("snippet","")[:300] + "...") if len(u.get("snippet",""))>300 else u.get("snippet","")
        lines.append(f"- **{os.path.basename(src) if '://' not in src else src}** — {snippet}")
    return "\n".join(lines)

with gr.Blocks(title="Personal AI Knowledge Worker — Advanced") as app:
    gr.Markdown("## Personal AI Knowledge Worker — **Advanced Connectors**")
    with gr.Tab("Local/Drive files"):
        with gr.Row():
            with gr.Column(scale=1):
                base_dir = gr.Textbox(label="Base folder (e.g., /content/drive/MyDrive/KnowledgeBase)", value="")
                uploads = gr.Files(label="Or upload files", type="filepath")
                chunk_size_local = gr.Slider(256, 2000, value=800, step=16, label="Chunk size")
                chunk_overlap_local = gr.Slider(0, 600, value=200, step=10, label="Chunk overlap")
                store_kind = gr.Radio(label="Vector store", choices=["chroma","faiss"], value="chroma")
                persist_dir = gr.Textbox(label="Chroma persist directory", value=DEFAULT_PERSIST_DIR)
                collection_name = gr.Textbox(label="Collection name", value="personal_kb_adv")
                embedder_kind = gr.Dropdown(label="Embeddings", value="Local (all-MiniLM-L6-v2)", choices=[
                    "Local (all-MiniLM-L6-v2)",
                    "Local (bge-small-en-v1.5)",
                    "OpenAI (text-embedding-3-small)",
                ])
                embedder_model = gr.Textbox(label="(Advanced) Custom embedding model id", value="")
                btn_index_local = gr.Button("Index Local/Uploaded Documents", variant="primary")
                index_status_local = gr.Textbox(label="Index status")
                num_chunks_local = gr.Number(label="# Chunks", value=None)
                btn_clear = gr.Button("Clear Index")
                clear_status = gr.Textbox(label="Clear status")
    with gr.Tab("Google Workspace"):
        gr.Markdown("**Authenticate** first by uploading your OAuth client JSON to `/content/google_client_secrets.json`, then the flow will prompt in the console when you run an ingestion.")
        with gr.Row():
            with gr.Column(scale=1):
                gws_query = gr.Textbox(label="Drive filename contains...", value="")
                include_docs = gr.Checkbox(label="Include Google Docs", value=True)
                include_sheets = gr.Checkbox(label="Include Sheets", value=True)
                include_slides = gr.Checkbox(label="Include Slides", value=True)
                gws_max_files = gr.Slider(1, 200, value=30, step=1, label="Max files")
                chunk_size_gws = gr.Slider(256, 2000, value=800, step=16, label="Chunk size")
                chunk_overlap_gws = gr.Slider(0, 600, value=200, step=10, label="Chunk overlap")
                btn_ingest_gws = gr.Button("Add from Google Workspace", variant="primary")
                status_gws = gr.Textbox(label="Ingestion status")
                num_chunks_gws = gr.Number(label="# Chunks added", value=None)
    with gr.Tab("Gmail"):
        gr.Markdown("**Authenticate** first by uploading your OAuth client JSON to `/content/google_client_secrets.json`, then the flow will prompt in the console when you run an ingestion.")
        with gr.Row():
            with gr.Column(scale=1):
                gmail_query = gr.Textbox(label="Gmail search query (e.g., from:alice after:2024/01/01)", value="")
                gmail_max = gr.Slider(1, 200, value=50, step=1, label="Max messages")
                chunk_size_gmail = gr.Slider(256, 2000, value=800, step=16, label="Chunk size")
                chunk_overlap_gmail = gr.Slider(0, 600, value=200, step=10, label="Chunk overlap")
                btn_ingest_gmail = gr.Button("Add from Gmail", variant="primary")
                status_gmail = gr.Textbox(label="Ingestion status")
                num_chunks_gmail = gr.Number(label="# Chunks added", value=None)
    with gr.Tab("Slack"):
        with gr.Row():
            with gr.Column(scale=1):
                slack_token = gr.Textbox(label="Slack Bot token (xoxb-...)", value=SLACK_BOT_TOKEN)
                slack_channel = gr.Textbox(label="Channel (name or ID, e.g., #general or C123...)", value="#general")
                oldest = gr.Textbox(label="Oldest (ISO date/time, optional)", value="")
                latest = gr.Textbox(label="Latest (ISO date/time, optional)", value="")
                slack_max = gr.Slider(1, 5000, value=1000, step=1, label="Max messages")
                chunk_size_slack = gr.Slider(256, 2000, value=800, step=16, label="Chunk size")
                chunk_overlap_slack = gr.Slider(0, 600, value=200, step=10, label="Chunk overlap")
                btn_ingest_slack = gr.Button("Add from Slack", variant="primary")
                status_slack = gr.Textbox(label="Ingestion status")
                num_chunks_slack = gr.Number(label="# Chunks added", value=None)
    with gr.Tab("Chat"):
        with gr.Row():
            with gr.Column(scale=1):
                use_frontier = gr.Checkbox(label="Use Frontier (OpenAI) instead of local LLM", value=False)
                frontier_model = gr.Textbox(label="Frontier model (OpenAI)", value="gpt-4o-mini")
                local_model = gr.Textbox(label="Local LLM (HF)", value=DEFAULT_LOCAL_LLM)
                temperature = gr.Slider(0.0, 1.5, value=0.2, step=0.05, label="Temperature")
                max_tokens = gr.Slider(64, 2048, value=512, step=32, label="Max new tokens")
                top_k = gr.Slider(1, 40, value=12, step=1, label="Top‑K chunks")
                show_prompt = gr.Checkbox(label="Show prompt preview & contexts", value=True)
                question = gr.Textbox(label="Your question", value="", lines=3)
                btn_ask = gr.Button("Ask", variant="primary")
            with gr.Column(scale=1):
                answer = gr.Markdown(label="Answer")
                warn = gr.Textbox(label="Warnings/Errors")
                prompt_prev = gr.Textbox(label="Prompt preview", lines=10)
                sources_md = gr.Markdown(label="Sources used")

    btn_index_local.click(
        fn=ingest_local,
        inputs=[base_dir, uploads, chunk_size_local, chunk_overlap_local, store_kind, persist_dir, collection_name, embedder_kind, embedder_model],
        outputs=[index_status_local, num_chunks_local]
    )
    btn_clear.click(
        fn=clear_index,
        inputs=[store_kind, persist_dir],
        outputs=[clear_status]
    )
    btn_ingest_gws.click(
        fn=ingest_gworkspace,
        inputs=[gws_query, include_docs, include_sheets, include_slides, gws_max_files, chunk_size_gws, chunk_overlap_gws],
        outputs=[status_gws, num_chunks_gws]
    )
    btn_ingest_gmail.click(
        fn=ingest_gmail,
        inputs=[gmail_query, gmail_max, chunk_size_gmail, chunk_overlap_gmail],
        outputs=[status_gmail, num_chunks_gmail]
    )
    btn_ingest_slack.click(
        fn=ingest_slack,
        inputs=[slack_token, slack_channel, oldest, latest, slack_max, chunk_size_slack, chunk_overlap_slack],
        outputs=[status_slack, num_chunks_slack]
    )
    def _ask_and_format(q, k, lm_path, use_f, fm, temp, mx, showp):
        ans, err, prompt, used = ask(q, k, lm_path, use_f, fm, temp, mx, showp)
        srcs = format_sources(used)
        return ans or "", err or "", prompt or "", srcs or ""
    btn_ask.click(
        fn=_ask_and_format,
        inputs=[question, top_k, local_model, use_frontier, frontier_model, temperature, max_tokens, show_prompt],
        outputs=[answer, warn, prompt_prev, sources_md]
    )

print("✅ UI ready. In Colab/Jupyter, run: app.queue(concurrency_count=2, max_size=64).launch(share=True, debug=True)")

### 17) Authenticate Google APIs (one‑time per session)
Upload your OAuth client JSON to `/content/google_client_secrets.json` (Desktop app). Then run:

In [ ]:
# Example: pre‑warm authentication for Drive/Docs/Sheets/Slides and Gmail
# from googleapiclient.discovery import build
# _ = ensure_google_auth(TOKEN_DRIVE, SCOPES_DRIVE)  # will prompt you in console
# _ = ensure_google_auth(TOKEN_GMAIL, SCOPES_GMAIL)  # will prompt you in console
# print("Google auth tokens saved.")

### 18) Launch the app

In [ ]:
# In Colab, use share=True for an external link:
import gradio as gr
gr.close_all()
# app.queue(concurrency_count=2, max_size=64).launch(share=True, debug=True)
app.launch(share=True, debug=True)

---
## Notes & guidance
- **Security**: Use read‑only scopes; tokens are stored in `/content` and are ephemeral in Colab. Remove tokens when done.
- **Google Workspace**: The Drive query filters by name (`name contains '...'`) and mimeTypes (Docs/Sheets/Slides). Adjust as needed, e.g., `modifiedTime > '2024-01-01T00:00:00'`.
- **Gmail**: Use Gmail search syntax (e.g., `from:alice subject:project after:2024/01/01`). HTML emails are converted to text with BeautifulSoup.
- **Slack**: Provide a **Bot token** with `channels:history` (and `groups:history` if private) scopes. Set channel by **name** (`#general`) or **ID** (`C123...`).
- **RAG tuning** (from Lesson 126): If answers miss facts, raise **Top‑K**, shrink **chunk size**, or increase **overlap**. More context often helps, but watch token limits.
- **Local-only mode**: Select local embeddings + local LLM to avoid sending text to external services.